In [17]:
# # Which topics does retrieval fail on?
# 
# We want to identify what kinds of questions the system struggles with by 
# cross-referencing retrieval performance with BERTopic topic clusters.
# 
# What we have:
# - BERTopic clusters: 20 semantic topics from clustering all 1140 FAQ questions
# - Generated test queries: 420 rephrased queries from the LLM (3 prompt strategies)
# - Retrieval results: Per-query success/failure from the best config (vector_cosine, 86.9%)
# 
# What this cell does:
# 1. Maps each generated query to a BERTopic topic (via the original FAQ it came from)
# 2. Runs vector retrieval on all 420 queries
# 3. Breaks down R@5 by topic to find which clusters are hardest

In [6]:
import json
from collections import Counter

with open('/home/admin/LLM/LLM/01/web/data_cleaning/data/processed/clean.jsonl') as f:
    all_docs = [json.loads(line) for line in f]

ml_docs = [d for d in all_docs if d['course'] == 'ml-zoomcamp']

# ── Just show what's there ───────────────────────────────────────────────────
print(f"ml-zoomcamp: {len(ml_docs)} questions\n")

# Show a sample from each section
from collections import defaultdict
by_section = defaultdict(list)
for d in ml_docs:
    by_section[d.get('section', '?')].append(d)

print("Sample questions from each section:")
for section, docs in sorted(by_section.items()):
    print(f"\n  [{section}] {len(docs)} questions")
    for d in docs[:3]:
        print(f"    {d['question'][:120]}")

ml-zoomcamp: 426 questions

Sample questions from each section:

  [general] 50 questions
    How do I submit homework?
    What’s new in the 2025 edition?
    Are Jupyter Notebooks used?

  [misc] 36 questions
    Why do I need to provide a train.py file when I already have the notebook.ipynb file?
    Loading the Image with PILLOW library and converting to numpy array
    Is a train.py file necessary when you have a train.ipynb file in your midterm project directory?

  [module-1] 23 questions
    Windows: WSL and VS Code
    Uploading the homework to Github

  [module-1-homework] 8 questions
    Floating Point Precision
    How to avoid Value errors with array shapes in homework?
    Homework Q5: How to replace NaNs with the average?

  [module-10] 30 questions
    What tools are recommended for setting up a local Kubernetes environment for model deployment practice?
    How do you determine the minimum number of replicas needed to guarantee zero-downtime for model updates in produc

In [7]:
import json, os, glob
from collections import defaultdict

# Load the latest variations results
files = sorted(glob.glob('/home/admin/LLM/LLM/01/web/experiments/results/variations_*.json'))
with open(files[-1]) as f:
    data = json.load(f)

# Find the best config
hybrid = next(c for c in data['configs'] if 'hybrid_70_30_vec' in c['name'] or 'hybrid' in c['name'])

# Or use vector_cosine if hybrid not found
if not hybrid:
    hybrid = next(c for c in data['configs'] if 'vector_cosine' in c['name'])

print(f"Config: {hybrid['name']}")
print(f"Total failures: {hybrid['failure_count']}")

# Load the eval queries to map back to original questions
with open('/home/admin/LLM/LLM/01/web/experiments/eval_queries.json') as f:
    qdata = json.load(f)

# Build query → original question mapping
query_map = {}
for doc in qdata['queries']:
    for strategy, variations in doc['prompt_results'].items():
        for v in variations:
            query_map[v] = {
                'original': doc['original_question'],
                'expected_id': doc['expected_id'],
                'course': doc['course'],
            }

# Show failures by course
if hybrid.get('failures_sample'):
    print(f"\nFailures by course:")
    by_course = defaultdict(list)
    for f in hybrid['failures_sample']:
        mapped = query_map.get(f['query'], {})
        course = mapped.get('course', 'unknown')
        by_course[course].append(f)
    
    for course, fails in sorted(by_course.items()):
        print(f"\n  [{course}] {len(fails)} failures")
        for f in fails[:5]:
            original = query_map.get(f['query'], {}).get('original', f['query'])
            print(f"    Query: {f['query'][:80]}")
            print(f"    Original FAQ: {original[:80]}")
            print(f"    Top sim: {f.get('top_sim', 'N/A')}")
            print()

Config: hybrid_rrf
Total failures: 43

Failures by course:

  [de-zoomcamp] 5 failures
    Query: installing dependencies for jupyter notebook
    Original FAQ: Other packages needed but not listed
    Top sim: 0.6110082268714905

    Query: encoding error when reading data from the web
    Original FAQ: Python: invalid start byte Error Message
    Top sim: 0.42706170678138733

    Query: csv file has weird bytes in it
    Original FAQ: Python: invalid start byte Error Message
    Top sim: 0.5184566378593445

    Query: i need to download something but my computer doesn't know what that is
    Original FAQ: wget is not recognized as an internal or external command
    Top sim: 0.20551477372646332

    Query: how do i install the downloader thing
    Original FAQ: wget is not recognized as an internal or external command
    Top sim: 0.3688296973705292



In [8]:
import json, re
from collections import Counter, defaultdict
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans

with open('/home/admin/LLM/LLM/01/web/data_cleaning/data/processed/clean.jsonl') as f:
    docs = [json.loads(line) for line in f]

questions = [d['question'] for d in docs]
courses = [d['course'] for d in docs]

# TF-IDF vectorization
vec = TfidfVectorizer(max_features=100, stop_words='english')
X = vec.fit_transform(questions)

# Cluster into groups
kmeans = KMeans(n_clusters=8, random_state=42)
labels = kmeans.fit_predict(X)

# Show top terms per cluster
print("=== Question Clusters ===\n")
order_centroids = kmeans.cluster_centers_.argsort()[:, ::-1]
terms = vec.get_feature_names_out()

for i in range(8):
    cluster_docs = [questions[j] for j in range(len(labels)) if labels[j] == i]
    cluster_courses = [courses[j] for j in range(len(labels)) if labels[j] == i]
    
    print(f"Cluster {i+1}: {len(cluster_docs)} questions")
    print(f"  Top terms: {', '.join([terms[idx] for idx in order_centroids[i, :8]])}")
    print(f"  Courses: {dict(Counter(cluster_courses))}")
    print(f"  Sample:")
    for q in cluster_docs[:3]:
        print(f"    {q[:100]}")
    print()

=== Question Clusters ===

Cluster 1: 17 questions
  Top terms: connection, server, vm, error, failed, aws, postgres, gcp
  Courses: {'de-zoomcamp': 6, 'llm-zoomcamp': 1, 'ml-zoomcamp': 4, 'mlops-zoomcamp': 6}
  Sample:
    Environment - Could not establish connection to "MyServerName": Got bad result from install script
    pgAdmin: persist server / connection settings across container restarts
    GCP VM: VM connection request timeout

Cluster 2: 57 questions
  Top terms: homework, submit, does, use, certificate, work, getting, dlt
  Courses: {'de-zoomcamp': 13, 'llm-zoomcamp': 5, 'ml-zoomcamp': 30, 'mlops-zoomcamp': 9}
  Sample:
    Homework: What are homework and project deadlines?
    Homework: Are late submissions of homework allowed?
    Homework: What is the homework URL in the homework link?

Cluster 3: 80 questions
  Top terms: docker, container, compose, image, error, running, run, install
  Courses: {'de-zoomcamp': 33, 'llm-zoomcamp': 2, 'ml-zoomcamp': 29, 'mlops-zoomcamp':

In [9]:
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
import json

# Load questions
with open('/home/admin/LLM/LLM/01/web/data_cleaning/data/processed/clean.jsonl') as f:
    docs = [json.loads(line) for line in f]

questions = [d['question'] for d in docs]

# BERTopic with your existing embedding model
embedding_model = SentenceTransformer('BAAI/bge-small-en-v1.5')
topic_model = BERTopic(embedding_model=embedding_model, min_topic_size=10, verbose=True)

topics, probs = topic_model.fit_transform(questions)

# Show results
print(f"\nFound {len(set(topics))-1} topics (+ noise)\n")
for topic_num in sorted(set(topics)):
    if topic_num == -1:
        continue
    topic_words = topic_model.get_topic(topic_num)
    doc_count = sum(1 for t in topics if t == topic_num)
    print(f"Topic {topic_num}: {doc_count} questions")
    print(f"  Keywords: {', '.join([w for w, _ in topic_words[:8]])}")
    # Show sample
    indices = [i for i, t in enumerate(topics) if t == topic_num][:3]
    for idx in indices:
        print(f"    {questions[idx][:100]}")
    print()

/home/admin/LLM/LLM/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 825.94it/s]
2026-05-12 10:43:40,605 - BERTopic - Embedding - Transforming documents to embeddings.
Batches: 100%|██████████| 36/36 [00:11<00:00,  3.12it/s]
2026-05-12 10:43:52,183 - BERTopic - Embedding - Completed ✓
2026-05-12 10:43:52,184 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-05-12 10:44:04,275 - BERTopic - Dimensionality - Completed ✓
2026-05-12 10:44:04,276 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-05-12 10:44:04,306 - BERTopic - Cluster - Completed ✓
2026-05-12 10:44:04,316 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-05-12 10:44:04,364 - BERTopic - Representation 


Found 20 topics (+ noise)

Topic 0: 304 questions
  Keywords: the, mlflow, homework, to, of, is, and, in
    Homework and Leaderboard: how does the points system work?
    RRPGCLI: Case sensitive use of “Quotations” around columns with capital letters
    GCP BQ ML - Export ML model to make predictions does not work for MacBook with Apple M1 chip (arm ar

Topic 1: 92 questions
  Keywords: pipenv, module, install, python, not, dependencies, version, file
    Environment: which Python version should I use?
    Environment - Could not establish connection to "MyServerName": Got bad result from install script
    How to run Python as a startup script?

Topic 2: 75 questions
  Keywords: docker, compose, container, failed, to, pgcli, daemon, running
    GitHub Codespaces: Running pgadmin in Docker
    Docker: Cannot connect to Docker daemon at unix:///var/run/docker.sock. Is the Docker daemon running
    Docker - error during connect: In the default daemon configuration on Windows, the dock

In [25]:
import json, os

# Check RAGAS progress
if os.path.exists('/home/admin/LLM/LLM/01/web/experiments/ragas_scores.json'):
    with open('/home/admin/LLM/LLM/01/web/experiments/ragas_scores.json') as f:
        data = json.load(f)
    scores = data.get('scores', {})
    completed = data.get('completed', [])
    deep = data.get('deep_completed', [])
    print(f"RAGAS scores: {len(scores)}")
    print(f"Fast completed: {len(completed)}")
    print(f"Deep completed: {len(deep)}")

RAGAS scores: 59
Fast completed: 20
Deep completed: 40


In [11]:
import json, glob
from collections import defaultdict
import numpy as np

# Load BERTopic model and variations results
from bertopic import BERTopic

# Load the topic model (we just fit it above)
# Map each question to its topic
topic_map = dict(zip(questions, topics))

# Load latest variations results
files = sorted(glob.glob('/home/admin/LLM/LLM/01/web/experiments/results/variations_*.json'))
with open(files[-1]) as f:
    var_data = json.load(f)

# Get the best config
best = max(var_data['configs'], key=lambda c: c['recall_at_k']['5'])

# Load eval queries to map back
with open('/home/admin/LLM/LLM/01/web/experiments/eval_queries.json') as f:
    qdata = json.load(f)

# Build query → original question → topic mapping
query_topic_map = {}
for doc in qdata['queries']:
    original_q = doc['original_question']
    topic = topic_map.get(original_q, -1)
    for strategy, variations in doc['prompt_results'].items():
        for v in variations:
            query_topic_map[v] = topic

# Now cross-reference failures
if best.get('failures_sample'):
    by_topic = defaultdict(lambda: {'fails': 0, 'total': 0})
    
    for f in best['failures_sample']:
        topic = query_topic_map.get(f['query'], -1)
        by_topic[topic]['fails'] += 1
    
    # Also count total queries per topic
    for q, topic in query_topic_map.items():
        by_topic[topic]['total'] += 1
    
    # Topic labels from BERTopic
    topic_labels = {}
    for topic_num in set(topics):
        if topic_num == -1:
            topic_labels[-1] = "Noise"
        else:
            words = topic_model.get_topic(topic_num)
            topic_labels[topic_num] = ', '.join([w for w, _ in words[:4]])
    
    print(f"Config: {best['name']} (R@5: {best['recall_at_k']['5']:.1%})\n")
    print(f"{'Topic':<40} {'Fails':>6} {'Total':>6} {'Rate':>8}")
    print(f"{'-'*40} {'-'*6} {'-'*6} {'-'*8}")
    
    for topic_num in sorted(by_topic.keys(), key=lambda t: by_topic[t]['fails'] / max(by_topic[t]['total'], 1), reverse=True):
        stats = by_topic[topic_num]
        rate = stats['fails'] / max(stats['total'], 1)
        label = topic_labels.get(topic_num, f"Topic {topic_num}")
        print(f"{label:<40} {stats['fails']:>6} {stats['total']:>6} {rate:>7.1%}")

Config: vector_cosine (R@5: 86.9%)

Topic                                     Fails  Total     Rate
---------------------------------------- ------ ------ --------
the, mlflow, homework, to                     3    144    2.1%
Noise                                         2     99    2.0%
docker, compose, container, failed            0     33    0.0%
terraform, googleapis, provider, com          0      9    0.0%
dbt, github, repository, how                  0     17    0.0%
aws, lambda, cli, creating                    0      9    0.0%
pipenv, module, install, python               0     39    0.0%
cloud, alternative, aws, alternatives         0      9    0.0%
grafana, mage, dashboard, chart               0     18    0.0%
openai, opensource, ai, non                   0      9    0.0%
course, cohort, the, can                      0      6    0.0%
gcp, vm, bucket, dataproc                     0     18    0.0%
project, projects, capstone, attempt          0      9    0.0%


In [15]:
import json, glob, os
from collections import defaultdict

# ── Reload everything ────────────────────────────────────────────────────────
files = sorted(glob.glob('/home/admin/LLM/LLM/01/web/experiments/results/variations_*.json'))
with open(files[-1]) as f:
    var_data = json.load(f)

best = max(var_data['configs'], key=lambda c: c['recall_at_k']['5'])
best_name = best['name']

# Load query → topic mapping (from earlier BERTopic cell — re-run if needed)
# topic_map was built from questions and topics

# Load raw results
raw_file = '/home/admin/LLM/LLM/01/web/experiments/results/vector_default.json'
with open(raw_file) as f:
    raw_data = json.load(f)

raw_results = [r for r in raw_data['results'] if r['k'] == 5]

# Map queries to topics
topic_stats = defaultdict(lambda: {'found': 0, 'total': 0})
for r in raw_results:
    query = r['query']
    topic = topic_map.get(query, -1) if 'topic_map' in dir() else -1
    topic_stats[topic]['total'] += 1
    if r['success']:
        topic_stats[topic]['found'] += 1

print(f"Config: {best_name} ({best['recall_at_k']['5']:.1%})\n")
print(f"{'Topic':<50} {'Found':>6} {'Total':>6} {'R@5':>8}")
print(f"{'-'*50} {'-'*6} {'-'*6} {'-'*8}")

for topic_num in sorted(topic_stats.keys(), key=lambda t: topic_stats[t]['found'] / max(topic_stats[t]['total'], 1)):
    stats = topic_stats[topic_num]
    rate = stats['found'] / max(stats['total'], 1)
    try:
        words = topic_model.get_topic(topic_num)
        label = ', '.join([w for w, _ in words[:4]])
    except:
        label = f"Topic {topic_num}"
    print(f"{label:<50} {stats['found']:>6} {stats['total']:>6} {rate:>7.1%}")

Config: vector_cosine (86.9%)

Topic                                               Found  Total      R@5
-------------------------------------------------- ------ ------ --------
project, projects, capstone, attempt                   23     23  100.0%
the, mlflow, homework, to                             305    305  100.0%
bigquery, bq, parquet, spark                           64     64  100.0%
dbt, github, repository, how                           47     47  100.0%
error, to, the, for                                   253    253  100.0%
kestra, why, in, workflow                              26     26  100.0%
taxi, data, yellow, fact_trips                         18     18  100.0%
pipenv, module, install, python                        93     93  100.0%
course, cohort, the, can                               35     35  100.0%
cloud, alternative, aws, alternatives                  10     10  100.0%
certificate, get, homeworks, homework                  13     13  100.0%
openai, opensource

In [18]:
import json, time
import numpy as np
from collections import defaultdict
from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient
from qdrant_client.models import Filter, FieldCondition, MatchValue

# ── Load data ────────────────────────────────────────────────────────────────
MODEL_NAME = 'BAAI/bge-base-en-v1.5'
QDRANT_COLLECTION = 'faqs_bge_base_en_v1.5'
TOP_K = 5

# Generated test queries
with open('/home/admin/LLM/LLM/01/web/experiments/eval_queries.json') as f:
    qdata = json.load(f)

# Build query list with topic labels from earlier BERTopic run
test_queries = []
for doc in qdata['queries']:
    original_q = doc['original_question']
    topic = topic_map.get(original_q, -1)  # From BERTopic cell
    for strategy, variations in doc['prompt_results'].items():
        for query in variations:
            test_queries.append({
                'query': query,
                'expected_id': doc['expected_id'],
                'course': doc['course'],
                'strategy': strategy,
                'topic': topic,
            })

print(f"Running retrieval on {len(test_queries)} queries...")

# ── Run retrieval ────────────────────────────────────────────────────────────
model = SentenceTransformer(MODEL_NAME)
client = QdrantClient('localhost', port=6333)

results = []
for i, tq in enumerate(test_queries):
    vec = model.encode(tq['query']).tolist()
    
    qfilter = Filter(must=[FieldCondition(key='course', match=MatchValue(value=tq['course']))])
    hits = client.query_points(collection_name=QDRANT_COLLECTION, query=vec, limit=TOP_K, query_filter=qfilter, with_payload=True)
    
    hit_ids = [h.payload.get('es_id', '') for h in hits.points]
    rank = next((pos for pos, hid in enumerate(hit_ids, 1) if hid == tq['expected_id']), None)
    
    results.append({
        'topic': tq['topic'],
        'strategy': tq['strategy'],
        'found': rank is not None,
        'rank': rank,
    })
    
    if (i+1) % 50 == 0:
        print(f"  {i+1}/{len(test_queries)}...")

# ── Results by topic ─────────────────────────────────────────────────────────
topic_labels = {}
for topic_num in set(tq['topic'] for tq in test_queries):
    if topic_num == -1:
        topic_labels[-1] = "Noise"
    else:
        try:
            words = topic_model.get_topic(topic_num)
            topic_labels[topic_num] = ', '.join([w for w, _ in words[:4]])
        except:
            topic_labels[topic_num] = f"Topic {topic_num}"

topic_stats = defaultdict(lambda: {'found': 0, 'total': 0})
for r in results:
    topic_stats[r['topic']]['total'] += 1
    if r['found']:
        topic_stats[r['topic']]['found'] += 1

total_found = sum(1 for r in results if r['found'])
print(f"\nOverall R@5: {total_found}/{len(results)} ({total_found/len(results):.1%})\n")
print(f"{'Topic':<50} {'Found':>6} {'Total':>6} {'R@5':>8}")
print(f"{'-'*50} {'-'*6} {'-'*6} {'-'*8}")

for topic_num in sorted(topic_stats.keys(), key=lambda t: topic_stats[t]['found'] / max(topic_stats[t]['total'], 1)):
    stats = topic_stats[topic_num]
    rate = stats['found'] / max(stats['total'], 1)
    label = topic_labels.get(topic_num, f"Topic {topic_num}")
    print(f"{label:<50} {stats['found']:>6} {stats['total']:>6} {rate:>7.1%}")

Running retrieval on 420 queries...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1017.19it/s]


  50/420...
  100/420...
  150/420...
  200/420...
  250/420...
  300/420...
  350/420...
  400/420...

Overall R@5: 376/420 (89.5%)

Topic                                               Found  Total      R@5
-------------------------------------------------- ------ ------ --------
course, cohort, the, can                                4      6   66.7%
openai, opensource, ai, non                             7      9   77.8%
the, mlflow, homework, to                             124    144   86.1%
docker, compose, container, failed                     29     33   87.9%
pipenv, module, install, python                        35     39   89.7%
Noise                                                  89     99   89.9%
grafana, mage, dashboard, chart                        17     18   94.4%
gcp, vm, bucket, dataproc                              17     18   94.4%
terraform, googleapis, provider, com                    9      9  100.0%
dbt, github, repository, how                           18    

In [19]:
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans

# Get all questions in Topic 0
topic0_indices = [i for i, t in enumerate(topics) if t == 0]
topic0_questions = [questions[i] for i in topic0_indices]

print(f"Sub-clustering {len(topic0_questions)} questions from Topic 0...")

# Embed and cluster
embeddings = embedding_model.encode(topic0_questions)
kmeans = KMeans(n_clusters=5, random_state=42)
sub_labels = kmeans.fit_predict(embeddings)

# Show clusters
for sub in range(5):
    indices = [i for i, l in enumerate(sub_labels) if l == sub]
    cluster_qs = [topic0_questions[i] for i in indices]
    print(f"\n  Sub-cluster {sub+1}: {len(cluster_qs)} questions")
    for q in cluster_qs[:5]:
        print(f"    {q[:120]}")

Sub-clustering 304 questions from Topic 0...

  Sub-cluster 1: 38 questions
    Kafka homework Q3: There are options that support the scaling concept more than the others.
    What are embeddings?
    Why was .dot(...) used directly to compute cosine similarity in the lesson, but normalization is emphasized in the homew
    Vector search: should I embed the question, the answer, or both?
    What is the cosine similarity?

  Sub-cluster 2: 63 questions
    Python: invalid start byte Error Message
    Python: Generators in Python
    Python: These won't work. You need to make sure you use Int64.
    The - vars argument must be a YAML dictionary, but was of type str
    MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory

  Sub-cluster 3: 73 questions
    RRPGCLI: Case sensitive use of “Quotations” around columns with capital letters
    Why do we need the Staging dataset?
    Homework: Ingesting FHV data from the course's GitHub mirror
    Homework: Inges

In [22]:
# Map each question to its sub-cluster
sub_cluster_map = {}
for i, sub in enumerate(sub_labels):
    q = topic0_questions[i]
    sub_cluster_map[q] = sub

# Now tag the test queries with sub-clusters and re-run stats
topic0_results = [r for r in results if r['topic'] == 0]
sub_stats = defaultdict(lambda: {'found': 0, 'total': 0})

for r in topic0_results:
    # Find which test query this is
    for tq in test_queries:
        if tq['query'] == r.get('query'):
            sub = sub_cluster_map.get(tq.get('original_q', ''), -1)
            sub_stats[sub]['total'] += 1
            if r['found']:
                sub_stats[sub]['found'] += 1
            break

print(f"\nTopic 0 sub-clusters R@5:")
for sub in range(5):
    stats = sub_stats[sub]
    rate = stats['found'] / max(stats['total'], 1)
    print(f"  Sub-cluster {sub+1}: {stats['found']}/{stats['total']} = {rate:.1%}")


Topic 0 sub-clusters R@5:
  Sub-cluster 1: 0/0 = 0.0%
  Sub-cluster 2: 0/0 = 0.0%
  Sub-cluster 3: 0/0 = 0.0%
  Sub-cluster 4: 0/0 = 0.0%
  Sub-cluster 5: 0/0 = 0.0%


In [23]:
# Build mapping from test query text → original FAQ question
query_to_original = {}
for doc in qdata['queries']:
    original_q = doc['original_question']
    for strategy, variations in doc['prompt_results'].items():
        for v in variations:
            query_to_original[v] = original_q

# Now re-run sub-cluster stats
sub_stats = defaultdict(lambda: {'found': 0, 'total': 0})
for r in results:
    if r['topic'] != 0:
        continue
    # Find the original question for this query
    for tq in test_queries:
        if tq['query'] == r.get('query'):
            original = query_to_original.get(tq['query'], '')
            sub = sub_cluster_map.get(original, -1)
            sub_stats[sub]['total'] += 1
            if r['found']:
                sub_stats[sub]['found'] += 1
            break

print(f"\nTopic 0 sub-clusters R@5:")
sub_labels_text = {
    0: "1. Vector search/embeddings",
    1: "2. Python errors/memory", 
    2: "3. Homework/data ingestion",
    3: "4. Hardware/Mac/GPU",
    4: "5. Leaderboard/scoring"
}
for sub in range(5):
    stats = sub_stats[sub]
    rate = stats['found'] / max(stats['total'], 1)
    bar = '█' * int(rate * 20)
    print(f"  {sub_labels_text.get(sub, f'Sub {sub}')}: {stats['found']}/{stats['total']} = {rate:.1%} {bar}")


Topic 0 sub-clusters R@5:
  1. Vector search/embeddings: 0/0 = 0.0% 
  2. Python errors/memory: 0/0 = 0.0% 
  3. Homework/data ingestion: 0/0 = 0.0% 
  4. Hardware/Mac/GPU: 0/0 = 0.0% 
  5. Leaderboard/scoring: 0/0 = 0.0% 


In [24]:
# Add sub-cluster to each test query before running retrieval
for tq in test_queries:
    original = query_to_original.get(tq['query'], '')
    tq['sub_topic'] = sub_cluster_map.get(original, -1)

# Now filter results for topic 0 and group by sub_topic
sub_stats = defaultdict(lambda: {'found': 0, 'total': 0})
for r in results:
    for tq in test_queries:
        if tq['query'] == r.get('query', ''):
            if tq['topic'] == 0:
                sub_stats[tq['sub_topic']]['total'] += 1
                if r['found']:
                    sub_stats[tq['sub_topic']]['found'] += 1
            break

print(f"Topic 0 sub-clusters R@5:")
for sub in range(5):
    stats = sub_stats[sub]
    total = stats['total']
    if total > 0:
        rate = stats['found'] / total
        print(f"  {sub_labels_text.get(sub, f'Sub {sub}')}: {stats['found']}/{total} = {rate:.1%}")
    else:
        print(f"  {sub_labels_text.get(sub, f'Sub {sub}')}: no test queries in this cluster")

Topic 0 sub-clusters R@5:
  1. Vector search/embeddings: no test queries in this cluster
  2. Python errors/memory: no test queries in this cluster
  3. Homework/data ingestion: no test queries in this cluster
  4. Hardware/Mac/GPU: no test queries in this cluster
  5. Leaderboard/scoring: no test queries in this cluster


In [26]:
import json, random
from collections import defaultdict

# Get Topic 0 questions by sub-cluster
sub_cluster_docs = defaultdict(list)
for i, sub in enumerate(sub_labels):
    q = topic0_questions[i]
    idx = topic0_indices[i]
    sub_cluster_docs[sub].append({
        'question': q,
        'id': all_docs[idx]['id'],  # from clean.jsonl
        'answer': all_docs[idx]['answer'],
        'course': all_docs[idx]['course'],
    })

# Sample 3 from each sub-cluster
random.seed(42)
sampled = []
for sub in range(5):
    docs = sub_cluster_docs[sub]
    sampled.extend(random.sample(docs, min(3, len(docs))))

print(f"Sampled {len(sampled)} questions from Topic 0 sub-clusters:")
for sub in range(5):
    count = sum(1 for s in sampled if sub_cluster_map.get(s['question']) == sub)
    print(f"  Sub-cluster {sub+1}: {count} questions")

Sampled 15 questions from Topic 0 sub-clusters:
  Sub-cluster 1: 3 questions
  Sub-cluster 2: 3 questions
  Sub-cluster 3: 3 questions
  Sub-cluster 4: 3 questions
  Sub-cluster 5: 3 questions


In [30]:
import json, random, re, os, time
from dotenv import load_dotenv
from litellm import completion as litellm_completion

load_dotenv('/home/admin/LLM/LLM/01/web/configs/.env')
os.environ["NVIDIA_NIM_API_KEY"] = os.getenv("NVIDIA_API_KEY")

MODEL = "nvidia_nim/meta/llama-3.1-70b-instruct"
PROMPT = """FAQ Question: {question}
FAQ Answer: {answer}

Write 3 different ways a student might search for this answer.
- Use different words than the original question
- Mix short keyword searches with full questions
- Include at least one that sounds like a real person typing quickly

Return ONLY a JSON array: ["query1", "query2", "query3"]"""

all_queries = []

for i, doc in enumerate(sampled):
    prompt = PROMPT.format(question=doc['question'], answer=doc['answer'][:400])
    
    for attempt in range(3):
        try:
            response = litellm_completion(model=MODEL, messages=[{"role":"user","content":prompt}], temperature=0.7, max_tokens=200)
            raw = response.choices[0].message.content.strip()
            raw = re.sub(r'```(?:json)?|```', '', raw).strip()
            variations = json.loads(raw)
            
            all_queries.append({
                'original_question': doc['question'],
                'expected_id': doc['id'],
                'course': doc['course'],
                'sub_topic': int(sub_cluster_map.get(doc['question'], -1)),
                'variations': variations[:3],
            })
            print(f"  [{i+1}/{len(sampled)}] {doc['question'][:60]}...")
            break
        except Exception as e:
            msg = str(e)
            if '429' in msg or '502' in msg or '504' in msg:
                wait = 60 * (attempt + 1)
                print(f"    Rate limit, waiting {wait}s...")
                time.sleep(wait)
            else:
                print(f"    Error: {e}")
                time.sleep(5)
    
    time.sleep(3)

# Save
def convert(o):
    import numpy as np
    if isinstance(o, np.integer): return int(o)
    if isinstance(o, np.floating): return float(o)
    return o

with open('/home/admin/LLM/LLM/01/web/experiments/topic0_queries.json', 'w') as f:
    json.dump({'metadata': {'count': len(all_queries)}, 'queries': all_queries}, f, indent=2, default=convert)

print(f"\nSaved {sum(len(q['variations']) for q in all_queries)} queries")

  [1/25] Singular Matrix Error...
  [2/25] What are embeddings?...
  [3/25] Understanding Ridge...
  [4/25] How do you find the correlation matrix?...
  [5/25] Homework Q4: Is r same as alpha in Scikit-Learn Ridge?...
  [6/25] ValueError: not enough values to unpack when parsing XGBoost...
  [7/25] I get `TypeError: got an unexpected keyword argument 'square...
    Error: Expecting value: line 1 column 1 (char 0)

Give Feedback / Get Help: https://github.com/BerriAI/litellm/issues/new
LiteLLM.Info: If you need to debug this error, use `litellm._turn_on_debug()'.


Provider List: https://docs.litellm.ai/docs/providers

    Rate limit, waiting 120s...
    Error: Expecting value: line 1 column 1 (char 0)
  [9/25] Error with the line interpreter.set_tensor(input_index, X)...
  [10/25] Error: Error while parsing arguments via CLI [ValueError: Un...
  [11/25] Overfitting: Absurdly high RMSE on the validation dataset...
  [12/25] Reproducibility: Do we have to run everything?...
  [13/25] Why

In [31]:
import json

with open('/home/admin/LLM/LLM/01/web/experiments/topic0_queries.json') as f:
    data = json.load(f)

print(f"Documents: {len(data['queries'])}")
print(f"Total queries: {sum(len(q['variations']) for q in data['queries'])}")

# Show by sub-topic
from collections import defaultdict
by_sub = defaultdict(list)
for q in data['queries']:
    by_sub[q['sub_topic']].append(q)

sub_names = {
    0: "Vector search/embeddings",
    1: "Python errors/memory", 
    2: "Homework/data ingestion",
    3: "Hardware/Mac/GPU",
    4: "Leaderboard/scoring"
}

for sub in sorted(by_sub.keys()):
    docs = by_sub[sub]
    print(f"\n{sub_names.get(sub, f'Sub {sub}')}: {len(docs)} documents, {sum(len(d['variations']) for d in docs)} queries")
    for d in docs[:2]:
        print(f"  {d['original_question'][:100]}")
        for v in d['variations'][:2]:
            print(f"    → {v}")

Documents: 24
Total queries: 72

Vector search/embeddings: 5 documents, 15 queries
  Singular Matrix Error
    → what is a singular matrix and how do i fix it
    → why cant i invert this matrix
  What are embeddings?
    → What is the process of turning non numerical data into numbers?
    → converting non numerical data to numerical data for machine learning

Python errors/memory: 4 documents, 12 queries
  ValueError: not enough values to unpack when parsing XGBoost output in parse_xg_output; how to fix a
    → xgboost error not enough values to unpack
    → Why is my xgboost model throwing a value error when parsing output?
  I get `TypeError: got an unexpected keyword argument 'squared'` when using `mean_squared_error(..., 
    → TypeError with mean squared error and squared=false
    → scikit learn rmse unexpected keyword

Homework/data ingestion: 5 documents, 15 queries
  Overfitting: Absurdly high RMSE on the validation dataset
    → high rmse on test set after removing outliers

In [32]:
import json, time
import numpy as np
from collections import defaultdict
from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient
from qdrant_client.models import Filter, FieldCondition, MatchValue

MODEL_NAME = 'BAAI/bge-base-en-v1.5'
QDRANT_COLLECTION = 'faqs_bge_base_en_v1.5'
TOP_K = 5

# Load queries
with open('/home/admin/LLM/LLM/01/web/experiments/topic0_queries.json') as f:
    t0_data = json.load(f)

# Flatten
test_queries = []
for doc in t0_data['queries']:
    for v in doc['variations']:
        test_queries.append({
            'query': v,
            'expected_id': doc['expected_id'],
            'course': doc['course'],
            'sub_topic': doc['sub_topic'],
        })

print(f"Running retrieval on {len(test_queries)} queries...")

model = SentenceTransformer(MODEL_NAME)
client = QdrantClient('localhost', port=6333)

results = []
for i, tq in enumerate(test_queries):
    vec = model.encode(tq['query']).tolist()
    qfilter = Filter(must=[FieldCondition(key='course', match=MatchValue(value=tq['course']))])
    hits = client.query_points(collection_name=QDRANT_COLLECTION, query=vec, limit=TOP_K, query_filter=qfilter, with_payload=True)
    
    hit_ids = [h.payload.get('es_id', '') for h in hits.points]
    rank = next((pos for pos, hid in enumerate(hit_ids, 1) if hid == tq['expected_id']), None)
    
    results.append({
        'sub_topic': tq['sub_topic'],
        'found': rank is not None,
        'rank': rank,
    })

# Results by sub-topic
sub_names = {0: "Vector search/embeddings", 1: "Python errors/memory", 2: "Homework/data ingestion", 3: "Hardware/Mac/GPU", 4: "Leaderboard/scoring"}

total_found = sum(1 for r in results if r['found'])
print(f"\nOverall R@5: {total_found}/{len(results)} ({total_found/len(results):.1%})\n")
print(f"{'Sub-topic':<30} {'Found':>6} {'Total':>6} {'R@5':>8}")
print(f"{'-'*30} {'-'*6} {'-'*6} {'-'*8}")

sub_stats = defaultdict(lambda: {'found': 0, 'total': 0})
for r in results:
    sub_stats[r['sub_topic']]['total'] += 1
    if r['found']:
        sub_stats[r['sub_topic']]['found'] += 1

for sub in sorted(sub_stats.keys()):
    stats = sub_stats[sub]
    rate = stats['found'] / max(stats['total'], 1)
    bar = '█' * int(rate * 20)
    print(f"{sub_names.get(sub, f'Sub {sub}'):<30} {stats['found']:>6} {stats['total']:>6} {rate:>7.1%} {bar}")

Running retrieval on 72 queries...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 12679.70it/s]



Overall R@5: 60/72 (83.3%)

Sub-topic                       Found  Total      R@5
------------------------------ ------ ------ --------
Vector search/embeddings           10     15   66.7% █████████████
Python errors/memory               12     12  100.0% ████████████████████
Homework/data ingestion            14     15   93.3% ██████████████████
Hardware/Mac/GPU                   14     15   93.3% ██████████████████
Leaderboard/scoring                10     15   66.7% █████████████


In [33]:
# Find the failed queries and what was returned
failures = []
for i, tq in enumerate(test_queries):
    if not results[i]['found']:
        vec = model.encode(tq['query']).tolist()
        qfilter = Filter(must=[FieldCondition(key='course', match=MatchValue(value=tq['course']))])
        hits = client.query_points(collection_name=QDRANT_COLLECTION, query=vec, limit=3, query_filter=qfilter, with_payload=True)
        
        top_hits = [(h.payload.get('question', '')[:100], h.payload.get('es_id', '')) for h in hits.points[:3]]
        
        failures.append({
            'query': tq['query'],
            'expected_id': tq['expected_id'],
            'sub_topic': sub_names.get(tq['sub_topic'], str(tq['sub_topic'])),
            'top3': top_hits,
        })

print(f"Failures: {len(failures)}\n")
# Group by sub-topic
from collections import defaultdict
by_sub = defaultdict(list)
for f in failures:
    by_sub[f['sub_topic']].append(f)

for sub, fails in by_sub.items():
    print(f"\n{'='*60}")
    print(f"{sub}: {len(fails)} failures")
    print(f"{'='*60}")
    for f in fails[:3]:
        print(f"\n  Query: {f['query']}")
        print(f"  Expected ID: {f['expected_id']}")
        print(f"  Top 3 returned:")
        for j, (q, qid) in enumerate(f['top3'], 1):
            match = " ← CORRECT" if qid == f['expected_id'] else ""
            print(f"    {j}. [{qid}] {q}{match}")

Failures: 12


Vector search/embeddings: 5 failures

  Query: What is the process of turning non numerical data into numbers?
  Expected ID: 69430a79a8
  Top 3 returned:
    1. [5beafdd0c8] I passed a float to my tool, but got a validation error saying it expected a number. Isn’t float a n
    2. [5039707c1a] How to normalize vectors in a Pandas DataFrame column (or Pandas Series)?
    3. [b4c8ac5a0c] How to compute the quantile or percentile of Pandas DataFrame column (or Pandas Series)?

  Query: converting non numerical data to numerical data for machine learning
  Expected ID: 69430a79a8
  Top 3 returned:
    1. [c1e9f4b820] Evaluation: hitting rate limits while generating the ground-truth dataset
    2. [5039707c1a] How to normalize vectors in a Pandas DataFrame column (or Pandas Series)?
    3. [b8f2d6a30c] Evaluation: Jupyter kernel crashes when embedding the ground-truth set

  Query: how do u turn words into numbers that a computer can understand??
  Expected ID: 69430a79a8
  